# Ứng dụng 1: Dự đoán Bệnh Tiểu đường (Diabetes Classification)
## Assignment 02 — Phát triển Hệ thống Thông minh (Intelligent System Development)

**Bài toán**: Phân loại nhị phân (Binary Classification) — Nhận diện bệnh nhân có nguy cơ mắc bệnh tiểu đường dựa trên các đặc điểm lâm sàng và sinh trắc học.

**Quy trình phát triển chuẩn công nghiệp (MLOps Lifecycle)**:
$$
\text{Raw Data} \rightarrow \text{Understand} \rightarrow \text{Clean} \rightarrow \text{Represent} \rightarrow \text{Learn} \rightarrow \text{Evaluate} \rightarrow \text{Persist} \rightarrow \text{Deploy}
$$

**Dataset**: `diabetes_dataset.csv` (100,000 mẫu thực tế từ Kaggle)

In [ ]:
# 0.1 Khai báo các thư viện cần thiết
import time
import pickle
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: Tiền xử lý, Mô hình, Tuning & Đánh giá
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)

print("Import toàn bộ thư viện thành công!")

In [ ]:
# 0.2 Cấu hình giao diện biểu đồ & Pandas hiển thị
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
print("Đã thiết lập cấu hình hiển thị biểu đồ và bảng dữ liệu.")

---
## Phần 1: Hiểu và Làm sạch Dữ liệu (Data Understanding & Cleaning)

**Mục tiêu**:
- Thấu hiểu phân phối, kiểu dữ liệu, các vấn đề tiềm ẩn trong dữ liệu thô (missing values, duplicated rows, outliers).
- Xử lý làm sạch có căn cứ khoa học/y học (WHY).

In [ ]:
# 1.1 Đọc tập dữ liệu thô
df = pd.read_csv('../../DATA/diabetes_dataset.csv')
print(f"Đọc dữ liệu thành công! Kích thước: {df.shape[0]:,} dòng × {df.shape[1]} cột")

In [ ]:
# 1.2 Kiểm tra kiểu dữ liệu các cột
df.dtypes

In [ ]:
# 1.3 Hiển thị 5 dòng đầu tiên
df.head(5)

In [ ]:
# 1.4 Thống kê mô tả số học (Descriptive Statistics)
df.describe()

In [ ]:
# 1.5 Kiểm tra số lượng giá trị bị thiếu (Missing Values)
missing_counts = df.isnull().sum()
print("Số lượng giá trị bị thiếu (Missing Values):")
missing_counts

In [ ]:
# 1.6 Kiểm tra số dòng trùng lặp (Duplicated Rows)
n_duplicated = df.duplicated().sum()
print(f"Số dòng trùng lặp hoàn toàn trong dataset: {n_duplicated}")

### Nhận xét & Kế hoạch làm sạch:
1. **Dòng trùng lặp**: Có 14 dòng trùng lặp $\rightarrow$ Cần loại bỏ để tránh bias khi phân chia tập dữ liệu.
2. **Cột không liên quan**: `year` (năm khảo sát - metadata, không phải triệu chứng y học) và `location` (55 địa điểm - cardinality cao dễ overfit) $\rightarrow$ Cần loại bỏ.
3. **Mã hóa (Encoding)**: `gender` (Label Encoding: Female=0, Male=1, Other=2) và `smoking_history` (Ordinal Encoding theo mức độ nguy cơ).
4. **Xử lý Ngoại lai (Outliers)**: Khảo sát IQR trên `bmi` và `blood_glucose_level` $\rightarrow$ Xử lý clipping hợp lý.

In [ ]:
# 1.7 Loại bỏ các dòng trùng lặp
df.drop_duplicates(inplace=True)
print(f"Đã loại bỏ trùng lặp! Kích thước hiện tại: {df.shape[0]:,} dòng × {df.shape[1]} cột")

*Lý do (WHY)*: Dòng trùng lặp làm mô hình học lặp lại cùng một điểm dữ liệu, gây thiên lệch trọng số và làm sai lệch chỉ số đánh giá nếu dòng trùng xuất hiện ở cả tập Train và Test.

In [ ]:
# 1.8 Loại bỏ các cột không mang tính chất lâm sàng
df.drop(columns=['year', 'location'], inplace=True)
print("Đã loại bỏ cột 'year' và 'location'. Danh sách cột còn lại:")
list(df.columns)

*Lý do (WHY)*: `year` là metadata thu thập dữ liệu (dễ gây temporal leakage), `location` không phải là cơ chế bệnh sinh của đái tháo đường mà chỉ là yếu tố hành chính.

In [ ]:
# 1.9 Mã hóa biến phân loại 'gender'
gender_map = {'Female': 0, 'Male': 1, 'Other': 2}
df['gender'] = df['gender'].map(gender_map)
print("Phân phối sau khi mã hóa cột 'gender':")
df['gender'].value_counts()

In [ ]:
# 1.10 Mã hóa biến có thứ bậc 'smoking_history' (Ordinal Encoding)
smoking_map = {
    'No Info': 0,      # Không có thông tin
    'never': 1,        # Chưa bao giờ hút
    'former': 2,       # Đã từng hút (đã bỏ)
    'not current': 3,  # Hiện không hút
    'ever': 4,         # Đã từng hút
    'current': 5       # Đang hút thuốc
}
df['smoking_history'] = df['smoking_history'].map(smoking_map)
print("Mã hóa 'smoking_history' theo cấp độ nguy cơ thành công:")
for k, v in smoking_map.items():
    print(f"  {k:15s} -> {v}")

*Lý do (WHY)*: Thuốc lá là yếu tố nguy cơ tim mạch - chuyển hóa có mức độ tăng dần theo cường độ tiếp xúc (`never` < `former` < `current`), do đó Ordinal Encoding phản ánh đúng bản chất hơn One-Hot Encoding.

In [ ]:
# 1.11 Kiểm tra ngoại lai (Outlier Detection) cho BMI bằng IQR
Q1_bmi = df['bmi'].quantile(0.25)
Q3_bmi = df['bmi'].quantile(0.75)
IQR_bmi = Q3_bmi - Q1_bmi
lower_bmi = Q1_bmi - 1.5 * IQR_bmi
upper_bmi = Q3_bmi + 1.5 * IQR_bmi

print(f"BMI - Q1: {Q1_bmi:.2f}, Q3: {Q3_bmi:.2f}, IQR: {IQR_bmi:.2f}")
print(f"Ngưỡng ngoại lai lý thuyết IQR: [{lower_bmi:.2f}, {upper_bmi:.2f}]")
print(f"Thực tế dataset: min = {df['bmi'].min():.2f}, max = {df['bmi'].max():.2f}")

In [ ]:
# 1.12 Xử lý ngoại lai BMI bằng phương pháp Clipping (Cắt ngưỡng lâm sàng)
# BMI dưới 10 hoặc trên 60 là bất thường y khoa cực hạn
n_clipped = ((df['bmi'] < 10) | (df['bmi'] > 60)).sum()
df['bmi'] = df['bmi'].clip(lower=10, upper=60)
print(f"Đã clip {n_clipped} giá trị BMI vào khoảng an toàn [10, 60]. Min={df['bmi'].min()}, Max={df['bmi'].max()}")

*Lý do (WHY)*: Sử dụng kỹ thuật **Winsorization / Clipping** thay vì xóa mẫu giúp bảo toàn kích thước dữ liệu và thông tin của bệnh nhân, đồng thời ngăn chặn các giá trị phi thực tế làm méo mó trọng số mô hình.

In [ ]:
# 1.13 Khảo sát ngoại lai cho Blood Glucose Level
print(f"Blood Glucose Level: min = {df['blood_glucose_level'].min()}, max = {df['blood_glucose_level'].max()}")
print("Giữ nguyên các mức glucose cao (200-300 mg/dL) vì đây là chỉ dấu bệnh lý đắt giá của bệnh nhân tiểu đường nặng.")

In [ ]:
# 1.14 Kiểm tra lại thông tin DataFrame sau khi hoàn tất làm sạch
df.info()

---
## Phần 2: Biểu diễn Dữ liệu (Data Representation)

**Mục tiêu**:
Trình bày cấu trúc toán học của ma trận đặc trưng $X \in \mathbb{R}^{N \times d}$ và vector mục tiêu $y \in \mathbb{R}^{N}$.

In [ ]:
# 2.1 Tách ma trận đặc trưng X và vector nhãn y
X = df.drop(columns=['diabetes'])
y = df['diabetes']

N, d = X.shape
print(f"Ma trận đặc trưng (Feature Matrix): X in R^({N:,} x {d})")
print(f"Vector mục tiêu (Target Vector):    y in R^({len(y):,})")

In [ ]:
# 2.2 Hiển thị 5 dòng đầu tiên của ma trận đặc trưng X
X.head()

In [ ]:
# 2.3 Bảng định nghĩa và kiểu biểu diễn toán học của các đặc trưng
feature_representation = pd.DataFrame({
    'Tên đặc trưng': X.columns,
    'Kiểu biểu diễn': ['Số nguyên (Encoded)', 'Số thực/nguyên', 'Nhị phân (OHE)', 'Nhị phân (OHE)',
                       'Nhị phân (OHE)', 'Nhị phân (OHE)', 'Nhị phân (OHE)', 'Nhị phân (0/1)',
                       'Nhị phân (0/1)', 'Số nguyên (Ordinal)', 'Số thực liên tục',
                       'Số thực liên tục', 'Số nguyên liên tục'],
    'Miền giá trị (Range)': [f"[{X[c].min()}, {X[c].max()}]" for c in X.columns],
    'Ý nghĩa lâm sàng': [
        'Giới tính sinh học', 'Độ tuổi bệnh nhân', 'Chủng tộc Da đen', 'Chủng tộc Châu Á',
        'Chủng tộc Da trắng', 'Chủng tộc Hispanic', 'Chủng tộc khác', 'Tiền sử tăng huyết áp',
        'Tiền sử bệnh tim mạch', 'Thói quen hút thuốc', 'Chỉ số khối cơ thể (BMI)',
        'Chỉ số đường huyết gắn Hemoglobin HbA1c (%)', 'Nồng độ đường trong máu (mg/dL)'
    ]
})
feature_representation

---
## Phần 3: Khám phá Dữ liệu (Exploratory Data Analysis — EDA)

Xây dựng ít nhất 3 biểu đồ trực quan hóa dữ liệu. Mỗi biểu đồ bắt buộc phải có đầy đủ 3 thành phần:
1. **Quan sát (Observation)**: Mô tả trực quan những gì thấy trên biểu đồ.
2. **Diễn giải (Interpretation)**: Giải thích bản chất y học / thống kê đằng sau.
3. **Ý nghĩa với Học máy (ML Implication)**: Chiến lược và lưu ý kỹ thuật khi xây dựng mô hình.

### Biểu đồ 1: Phân phối Biến mục tiêu (Target Distribution)

In [ ]:
# 3.1 Trực quan hóa phân phối biến mục tiêu diabetes
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

target_counts = y.value_counts().sort_index()
colors = ['#2ecc71', '#e74c3c']

# Barplot
bars = axes[0].bar(['Không bệnh (0)', 'Mắc bệnh (1)'], target_counts.values, color=colors, edgecolor='black', width=0.55)
for bar in bars:
    h = bar.get_height()
    axes[0].annotate(f'{h:,}\n({h/len(y)*100:.1f}%)',
                     xy=(bar.get_x() + bar.get_width() / 2, h),
                     xytext=(0, 5), textcoords="offset points",
                     ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[0].set_title('Phân phối số lượng nhãn Diabetes', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Số lượng mẫu (samples)')
axes[0].set_ylim(0, max(target_counts.values) * 1.15)

# Pie chart
axes[1].pie(target_counts.values, labels=['Không bệnh (0)', 'Mắc bệnh (1)'],
            colors=colors, autopct='%1.1f%%', startangle=140, explode=[0, 0.1], shadow=True)
axes[1].set_title('Tỷ lệ phần trăm giữa 2 lớp', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

#### Phân tích Biểu đồ 1:
- 🔍 **Quan sát (Observation)**: Tập dữ liệu gồm 91,486 bệnh nhân không mắc bệnh (91.5%) và 8,500 bệnh nhân mắc bệnh (8.5%). Tỷ lệ mất cân bằng khoảng $10.8 : 1$.
- 💡 **Diễn giải (Interpretation)**: Tỷ lệ 8.5% phản ánh sát thực tế tỷ lệ lưu hành bệnh đái tháo đường trong quần thể dân số nói chung.
- ⚙️ **Ý nghĩa với Học máy (ML Implication)**:
  - Đây là bài toán **Imbalanced Classification**. Không được sử dụng `Accuracy` làm chỉ số quyết định vì mô hình đoán toàn bộ 0 cũng đạt $91.5\%$ Accuracy.
  - Bắt buộc kích hoạt tham số `class_weight='balanced'` để phạt nặng hơn lỗi đoán sai lớp thiểu số.
  - Đánh giá bằng **F1-Score, Recall, Precision** và **ROC-AUC**.

### Biểu đồ 2: Phân phối Đặc trưng HbA1c và Blood Glucose theo tình trạng bệnh

In [ ]:
# 3.2 Phân phối của HbA1c và Glucose phân tách theo nhãn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# HbA1c
for cls, col, lbl in [(0, '#2ecc71', 'Không bệnh (0)'), (1, '#e74c3c', 'Mắc bệnh (1)')]:
    subset = df[df['diabetes'] == cls]['hbA1c_level']
    axes[0].hist(subset, bins=30, alpha=0.6, color=col, label=lbl, density=True, edgecolor='white')
axes[0].axvline(x=6.5, color='black', linestyle='--', linewidth=2, label='Ngưỡng chẩn đoán ADA (6.5%)')
axes[0].set_title('Phân phối chỉ số HbA1c (%)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('HbA1c level (%)')
axes[0].set_ylabel('Mật độ (Density)')
axes[0].legend()

# Blood Glucose
for cls, col, lbl in [(0, '#2ecc71', 'Không bệnh (0)'), (1, '#e74c3c', 'Mắc bệnh (1)')]:
    subset = df[df['diabetes'] == cls]['blood_glucose_level']
    axes[1].hist(subset, bins=30, alpha=0.6, color=col, label=lbl, density=True, edgecolor='white')
axes[1].axvline(x=126, color='black', linestyle='--', linewidth=2, label='Ngưỡng chẩn đoán đói (126 mg/dL)')
axes[1].set_title('Phân phối nồng độ Blood Glucose (mg/dL)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Blood Glucose (mg/dL)')
axes[1].set_ylabel('Mật độ (Density)')
axes[1].legend()

plt.tight_layout()
plt.show()

#### Phân tích Biểu đồ 2:
- 🔍 **Quan sát (Observation)**: Người mắc tiểu đường (màu đỏ) tập trung chủ yếu ở mức $HbA1c \ge 6.5\%$ và $Glucose > 140\text{ mg/dL}$. Ngược lại, người khỏe mạnh tập trung ở $HbA1c < 6.0\%$ và $Glucose < 120\text{ mg/dL}$.
- 💡 **Diễn giải (Interpretation)**: Hoàn toàn trùng khớp với tiêu chuẩn chẩn đoán quốc tế ADA (American Diabetes Association). Đây là 2 đặc trưng phân định sinh học trực tiếp nhất.
- ⚙️ **Ý nghĩa với Học máy (ML Implication)**:
  - 2 đặc trưng này chắc chắn sẽ chiếm trọng số / **Feature Importance cao nhất**.
  - Các thuật toán dạng cây (Decision Tree, Random Forest) sẽ chọn các ngưỡng này ở các node phân nhánh đầu tiên.

### Biểu đồ 3: Ma trận Tương quan Pearson (Correlation Matrix)

In [ ]:
# 3.3 Ma trận tương quan giữa tất cả các đặc trưng và nhãn
fig, ax = plt.subplots(figsize=(12, 9))

corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-0.2, vmax=0.6, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Hệ số tương quan Pearson'}, ax=ax)
ax.set_title('Ma trận Tương quan (Correlation Heatmap)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

#### Phân tích Biểu đồ 3:
- 🔍 **Quan sát (Observation)**: `blood_glucose_level` ($r \approx 0.42$) và `hbA1c_level` ($r \approx 0.40$) tương quan dương mạnh nhất với `diabetes`. `age` ($r \approx 0.26$) và `bmi` ($r \approx 0.21$) có mức tương quan vừa phải. Các đặc trưng còn lại tương quan thấp.
- 💡 **Diễn giải (Interpretation)**: Không có cặp đặc trưng đầu vào nào có hệ số tương quan $> 0.8$, chứng tỏ **không tồn tại hiện tượng đa cộng tuyến nghiêm trọng (Multicollinearity)**.
- ⚙️ **Ý nghĩa với Học máy (ML Implication)**:
  - Không cần áp dụng kỹ thuật giảm chiều (như PCA) vì mỗi đặc trưng đều đóng góp chiều thông tin độc lập.
  - Các mô hình tuyến tính (như Logistic Regression) sẽ hội tụ ổn định.

---
## Phần 4: Phát triển và Đánh giá Mô hình (Model Development & Evaluation)

### 4.1 Phân chia tập dữ liệu (Train / Validation / Test Split)
Phân chia theo tỷ lệ **70% Train / 15% Validation / 15% Test** với phương pháp phân tầng (`stratify=y`) để duy trì tỷ lệ lớp cân bằng trong từng tập.

In [ ]:
# 4.1.1 Thực hiện phân chia 70/15/15 có phân tầng (Stratified Split)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Kích thước tập Train:      {X_train.shape[0]:,} mẫu (70%)")
print(f"Kích thước tập Validation: {X_val.shape[0]:,} mẫu (15%)")
print(f"Kích thước tập Test:       {X_test.shape[0]:,} mẫu (15%)")

In [ ]:
# 4.1.2 Kiểm tra tỷ lệ lớp dương tính (Class 1) trên các tập
print(f"Tỷ lệ lớp 1 trong Train:      {y_train.mean()*100:.2f}%")
print(f"Tỷ lệ lớp 1 trong Validation: {y_val.mean()*100:.2f}%")
print(f"Tỷ lệ lớp 1 trong Test:       {y_test.mean()*100:.2f}%")

### 4.2 Xây dựng Pipeline Tiền xử lý (ColumnTransformer) — Chống Rò rỉ Dữ liệu (Data Leakage)

In [ ]:
# 4.2.1 Phân loại nhóm đặc trưng
continuous_features = ['age', 'bmi', 'hbA1c_level', 'blood_glucose_level']
passthrough_features = [col for col in X.columns if col not in continuous_features]

# Khởi tạo Transformer chuẩn hóa
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), continuous_features),
        ('pass', 'passthrough', passthrough_features)
    ]
)

# ⚠️ NGUYÊN TẮC VÀNG: FIT DUY NHẤT TRÊN TẬP TRAIN ĐỂ CHỐNG DATA LEAKAGE
preprocessor.fit(X_train)
print("Preprocessor đã được FIT thành công CHỈ trên tập Train.")

In [ ]:
# 4.2.2 Thực hiện Transform dữ liệu sang ma trận đặc trưng số học
X_train_scaled = preprocessor.transform(X_train)
X_val_scaled = preprocessor.transform(X_val)
X_test_scaled = preprocessor.transform(X_test)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_val_scaled shape:   {X_val_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")

### 4.3 Định nghĩa Hàm Đánh giá Mô hình Tổng thể

In [ ]:
# 4.3.1 Xây dựng hàm đánh giá chi tiết
def evaluate_model(model, X_eval, y_eval, model_name="Model", train_time=0.0):
    """Đánh giá mô hình phân loại và trả về dictionary kết quả."""
    y_pred = model.predict(X_eval)
    
    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)
    
    roc_auc = None
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_eval)[:, 1]
        roc_auc = roc_auc_score(y_eval, y_proba)
        
    print(f"[{model_name}] Thời gian huấn luyện: {train_time:.2f}s")
    print(f"  Accuracy:  {acc:.4f} | Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f} | F1-Score:  {f1:.4f} | ROC-AUC: {roc_auc if roc_auc is not None else 'N/A':.4f}")
    print("-" * 65)
    
    return {
        'Model': model_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(roc_auc, 4) if roc_auc is not None else 'N/A',
        'Train Time (s)': round(train_time, 2)
    }

print("Đã định nghĩa hàm evaluate_model().")

### 4.4 Huấn luyện và Dò tìm Siêu tham số (Hyperparameter Tuning) với GridSearchCV

Sử dụng `GridSearchCV` với $k=5$ fold cross-validation và hàm mục tiêu tối ưu `scoring='f1'`.

#### Mô hình 1: Logistic Regression (GridSearchCV)

In [ ]:
# 4.4.1 Cấu hình lưới tham số cho Logistic Regression
lr_param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'solver': ['lbfgs']
}

lr_base = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_grid = GridSearchCV(estimator=lr_base, param_grid=lr_param_grid,
                       cv=5, scoring='f1', n_jobs=-1)

# Đo lường thời gian huấn luyện
t_start = time.time()
lr_grid.fit(X_train_scaled, y_train)
lr_train_time = time.time() - t_start

print(f"Logistic Regression GridSearch hoàn tất trong {lr_train_time:.2f} giây!")
print(f"Tham số tối ưu nhất: {lr_grid.best_params_}")
print(f"F1-Score trung bình qua 5-Fold CV: {lr_grid.best_score_:.4f}")

In [ ]:
# 4.4.2 Đánh giá mô hình Logistic Regression tối ưu trên tập Validation
best_lr = lr_grid.best_estimator_
results_lr = evaluate_model(best_lr, X_val_scaled, y_val, "Logistic Regression (Tuned)", lr_train_time)

#### Mô hình 2: Decision Tree (GridSearchCV)

In [ ]:
# 4.4.3 Cấu hình lưới tham số cho Decision Tree
dt_param_grid = {
    'max_depth': [4, 6, 8, 10],
    'min_samples_split': [5, 10, 20],
    'criterion': ['gini', 'entropy']
}

dt_base = DecisionTreeClassifier(class_weight='balanced', random_state=42)
dt_grid = GridSearchCV(estimator=dt_base, param_grid=dt_param_grid,
                       cv=5, scoring='f1', n_jobs=-1)

t_start = time.time()
dt_grid.fit(X_train_scaled, y_train)
dt_train_time = time.time() - t_start

print(f"Decision Tree GridSearch hoàn tất trong {dt_train_time:.2f} giây!")
print(f"Tham số tối ưu nhất: {dt_grid.best_params_}")
print(f"F1-Score trung bình qua 5-Fold CV: {dt_grid.best_score_:.4f}")

In [ ]:
# 4.4.4 Đánh giá mô hình Decision Tree tối ưu trên tập Validation
best_dt = dt_grid.best_estimator_
results_dt = evaluate_model(best_dt, X_val_scaled, y_val, "Decision Tree (Tuned)", dt_train_time)

#### Mô hình 3: Random Forest (GridSearchCV)

In [ ]:
# 4.4.5 Cấu hình lưới tham số cho Random Forest
rf_param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [8, 12],
    'min_samples_split': [5, 10]
}

rf_base = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
rf_grid = GridSearchCV(estimator=rf_base, param_grid=rf_param_grid,
                       cv=5, scoring='f1', n_jobs=-1)

t_start = time.time()
rf_grid.fit(X_train_scaled, y_train)
rf_train_time = time.time() - t_start

print(f"Random Forest GridSearch hoàn tất trong {rf_train_time:.2f} giây!")
print(f"Tham số tối ưu nhất: {rf_grid.best_params_}")
print(f"F1-Score trung bình qua 5-Fold CV: {rf_grid.best_score_:.4f}")

In [ ]:
# 4.4.6 Đánh giá mô hình Random Forest tối ưu trên tập Validation
best_rf = rf_grid.best_estimator_
results_rf = evaluate_model(best_rf, X_val_scaled, y_val, "Random Forest (Tuned)", rf_train_time)

#### Mô hình 4: K-Nearest Neighbors (GridSearchCV quét qua dải $K$)

In [ ]:
# 4.4.7 Cấu hình dải tham số K đa dạng [3, 5, 7, 9, 11, 15, 21] cho KNN
knn_param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 15, 21],
    'weights': ['uniform', 'distance']
}

knn_base = KNeighborsClassifier(n_jobs=-1)
knn_grid = GridSearchCV(estimator=knn_base, param_grid=knn_param_grid,
                        cv=5, scoring='f1', n_jobs=-1)

t_start = time.time()
knn_grid.fit(X_train_scaled, y_train)
knn_train_time = time.time() - t_start

print(f"KNN GridSearch hoàn tất trong {knn_train_time:.2f} giây!")
print(f"Tham số tối ưu nhất: {knn_grid.best_params_}")
print(f"F1-Score trung bình qua 5-Fold CV: {knn_grid.best_score_:.4f}")

In [ ]:
# 4.4.8 Đánh giá mô hình KNN tối ưu trên tập Validation
best_knn = knn_grid.best_estimator_
results_knn = evaluate_model(best_knn, X_val_scaled, y_val, "KNN (Tuned)", knn_train_time)

#### Mô hình 5: Support Vector Machine (SVM - RBF Kernel)
*Lưu ý kỹ thuật*: Với kích thước mẫu lớn (~70k dòng train), thuật toán SVM giải bài toán tối ưu bậc 2 $O(N^2 \sim N^3)$, do đó ta cấu hình tĩnh tham số `C=1.0`, `kernel='rbf'`, `class_weight='balanced'` để đảm bảo thời gian chạy tối ưu.

In [ ]:
# 4.4.9 Huấn luyện SVM với tham số cấu hình riêng biệt
svm_model = SVC(C=1.0, kernel='rbf', gamma='scale',
                class_weight='balanced', probability=True, random_state=42)

t_start = time.time()
# Huấn luyện trên tập train
svm_model.fit(X_train_scaled, y_train)
svm_train_time = time.time() - t_start

print(f"SVM huấn luyện thành công trong {svm_train_time:.2f} giây!")

In [ ]:
# 4.4.10 Đánh giá mô hình SVM trên tập Validation
results_svm = evaluate_model(svm_model, X_val_scaled, y_val, "SVM (RBF Kernel)", svm_train_time)

---
### 4.5 Bảng Tổng hợp và So sánh Toàn diện 5 Mô hình

In [ ]:
# 4.5.1 Tổng hợp các chỉ số đánh giá lên DataFrame
comparison_df = pd.DataFrame([results_lr, results_dt, results_rf, results_knn, results_svm])
comparison_df.sort_values(by='F1-Score', ascending=False, inplace=True)
comparison_df.reset_index(drop=True, inplace=True)

print("=" * 80)
print("                     BẢNG SO SÁNH HIỆU SUẤT MÔ HÌNH (VALIDATION SET)")
print("=" * 80)
comparison_df

---
### 4.6 Khai phá Mức độ Quan trọng của Đặc trưng (Feature Importance)

Trích xuất thuộc tính `feature_importances_` từ mô hình Random Forest tốt nhất thu được sau GridSearchCV và trực quan hóa bằng horizontal barplot.

In [ ]:
# 4.6.1 Lấy danh sách tên đặc trưng theo đúng thứ tự trong ColumnTransformer
feature_names = continuous_features + passthrough_features

# Trích xuất tầm quan trọng đặc trưng từ best Random Forest
importances = best_rf.feature_importances_

# Tạo DataFrame mức độ quan trọng
df_importance = pd.DataFrame({
    'Đặc trưng': feature_names,
    'Mức độ quan trọng (Importance)': importances
}).sort_values(by='Mức độ quan trọng (Importance)', ascending=False)

df_importance.reset_index(drop=True, inplace=True)
df_importance

In [ ]:
# 4.6.2 Trực quan hóa Feature Importance bằng Horizontal Barplot (Seaborn)
fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=df_importance,
    x='Mức độ quan trọng (Importance)',
    y='Đặc trưng',
    palette='viridis',
    ax=ax,
    edgecolor='black',
    linewidth=0.5
)

# Thêm giá trị cụ thể ở đuôi mỗi thanh
for i, v in enumerate(df_importance['Mức độ quan trọng (Importance)']):
    ax.text(v + 0.005, i, f'{v*100:.2f}%', va='center', fontweight='bold', fontsize=10)

ax.set_title('Mức độ Quan trọng của các Đặc trưng (Random Forest Feature Importance)', fontsize=14, fontweight='bold')
ax.set_xlabel('Tỷ lệ đóng góp (Relative Importance)', fontsize=12)
ax.set_ylabel('Đặc trưng lâm sàng (Features)', fontsize=12)
ax.set_xlim(0, max(importances) * 1.18)

plt.tight_layout()
plt.show()

#### Phân tích Biểu đồ Feature Importance:
- 🔍 **Quan sát (Observation)**: `hbA1c_level` và `blood_glucose_level` áp đảo toàn bộ các đặc trưng khác, chiếm tổng cộng hơn $70\%$ tầm quan trọng của mô hình. Kế tiếp là `age` và `bmi`. Các đặc trưng nhân khẩu học (`race:*`, `gender`) có mức đóng góp rất thấp ($< 2\%$).
- 💡 **Diễn giải (Interpretation)**: HbA1c và Glucose phản ánh trạng thái rối loạn dung nạp glucose trực tiếp trong cơ thể, trong khi chủng tộc chỉ là yếu tố dịch tễ học phụ trợ.
- ⚙️ **Ý nghĩa với Học máy (ML Implication)**:
  - Khẳng định tính đúng đắn và khả năng giải thích (Explainability/Interpretability) của mô hình Random Forest đối với giới chuyên môn y khoa.
  - Khi triển khai ứng dụng thực tế, bắt buộc người dùng phải cung cấp chính xác 2 chỉ số này để có kết quả tin cậy.

---
### 4.7 Đánh giá Mô hình Tốt nhất trên Tập Kiểm thử (Final Test Evaluation)

In [ ]:
# 4.7.1 Gán mô hình tốt nhất vào biến 'best_model'
# Lựa chọn mô hình đạt F1-Score cao nhất trên tập Validation
top_model_name = comparison_df.iloc[0]['Model']
models_pool = {
    'Logistic Regression (Tuned)': best_lr,
    'Decision Tree (Tuned)': best_dt,
    'Random Forest (Tuned)': best_rf,
    'KNN (Tuned)': best_knn,
    'SVM (RBF Kernel)': svm_model
}

best_model = models_pool[top_model_name]
best_model_name = top_model_name

print(f"Mô hình được chọn làm Dịch vụ Triển khai: '{best_model_name}'")

In [ ]:
# 4.7.2 Đánh giá lần cuối và duy nhất trên tập TEST
y_test_pred = best_model.predict(X_test_scaled)
y_test_proba = best_model.predict_proba(X_test_scaled)[:, 1]

print("=" * 65)
print(f"      KẾT QUẢ ĐÁNH GIÁ LẦN CUỐI TRÊN TẬP TEST ({best_model_name})")
print("=" * 65)
print(f"  Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"  Recall:    {recall_score(y_test, y_test_pred):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_test_pred):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_test_proba):.4f}")
print("-" * 65)
print("Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=['Không bệnh (0)', 'Mắc bệnh (1)']))

In [ ]:
# 4.7.3 Vẽ Ma trận Nhầm lẫn (Confusion Matrix) trên tập Test
fig, ax = plt.subplots(figsize=(7, 5.5))

cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Dự đoán: Khỏe (0)', 'Dự đoán: Bệnh (1)'],
            yticklabels=['Thực tế: Khỏe (0)', 'Thực tế: Bệnh (1)'],
            annot_kws={'size': 16, 'fontweight': 'bold'},
            linewidths=1, linecolor='white')

ax.set_title(f'Confusion Matrix — {best_model_name} (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negative  (TN): {tn:>6,} | Dự đoán đúng người không bệnh.")
print(f"False Positive (FP): {fp:>6,} | Dự đoán nhầm người khỏe thành mắc bệnh.")
print(f"False Negative (FN): {fn:>6,} | Bỏ sót bệnh nhân thực tế (Nguy hiểm trong y tế!).")
print(f"True Positive  (TP): {tp:>6,} | Phát hiện chính xác bệnh nhân mắc tiểu đường.")

In [ ]:
# 4.7.4 Vẽ Đường cong ROC (ROC Curve)
fig, ax = plt.subplots(figsize=(8, 6))

fpr, tpr, _ = roc_curve(y_test, y_test_proba)
auc_val = roc_auc_score(y_test, y_test_proba)

ax.plot(fpr, tpr, color='#2980b9', lw=2.5, label=f'{best_model_name} (AUC = {auc_val:.4f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Phân loại ngẫu nhiên (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.15, color='#2980b9')

ax.set_title('Đường cong Đặc trưng Hoạt động của Bộ thu (ROC Curve)', fontsize=14, fontweight='bold')
ax.set_xlabel('Tỷ lệ Dương tính Giả (False Positive Rate - FPR)', fontsize=12)
ax.set_ylabel('Tỷ lệ Dương tính Thật (True Positive Rate - Recall)', fontsize=12)
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.05])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Phần 5: Lưu trữ và Triển khai (Persistence & Deployment)

Quy trình triển khai dịch vụ thông minh bao gồm:
1. **Lưu trữ (Persistence)**: Đóng gói toàn bộ Pipeline (Preprocessor + Model) vào các định dạng chuẩn `.joblib` và `.sav` (pickle).
2. **REST API Micro-service (Flask)**: Xây dựng endpoint chuẩn `/diabetes/v1/predict` theo tài liệu giáo trình.
3. **Kiểm thử API (Testing)**: Thử nghiệm với `cURL` và viết ứng dụng Client Python (`requests`).
4. **Giao diện Web Tương tác (FastAPI + Bootstrap 5)**: Hỗ trợ người dùng thao tác trực quan.

### 5.1 Đóng gói và Lưu trữ Pipeline (Joblib & Pickle)

In [ ]:
# 5.1.1 Xây dựng Full Pipeline hoàn chỉnh
os.makedirs('models', exist_ok=True)

# Đóng gói Preprocessor + Best Model
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', best_model)
])

# Đóng gói kèm Metadata
artifact = {
    'pipeline': full_pipeline,
    'gender_map': gender_map,
    'smoking_map': smoking_map,
    'feature_columns': list(X.columns),
    'continuous_features': continuous_features,
    'passthrough_features': passthrough_features,
    'model_name': best_model_name
}

# Lưu dưới dạng Joblib
joblib_path = 'models/diabetes_pipeline.joblib'
joblib.dump(artifact, joblib_path)
print(f"[1] Đã lưu Joblib artifact: '{joblib_path}' ({os.path.getsize(joblib_path)/1024:.1f} KB)")

# Lưu mô hình dạng .sav (pickle) theo chuẩn giáo trình
sav_path = 'models/diabetes.sav'
with open(sav_path, 'wb') as f:
    pickle.dump(artifact, f)
print(f"[2] Đã lưu Pickle model:    '{sav_path}' ({os.path.getsize(sav_path)/1024:.1f} KB)")

In [ ]:
# 5.1.2 Kiểm thử Nạp lại (Load Test) Pipeline
loaded_artifact = joblib.load('models/diabetes_pipeline.joblib')
loaded_pipe = loaded_artifact['pipeline']

# Dự đoán mẫu thử
sample_test = X_test.head(3)
sample_preds = loaded_pipe.predict(sample_test)
sample_probas = loaded_pipe.predict_proba(sample_test)

print("Kiểm thử nạp và suy luận thành công:")
for i in range(len(sample_test)):
    print(f"  Mẫu {i+1}: Dự đoán nhãn = {sample_preds[i]} | Độ tin cậy = {np.amax(sample_probas[i])*100:.2f}%")

### 5.2 Triển khai REST API với Flask Framework (Theo hướng dẫn Giáo trình)

Dưới đây là mã nguồn xây dựng dịch vụ REST API bằng Flask micro-framework, lắng nghe tại cổng `5000` với endpoint `/diabetes/v1/predict`.

In [ ]:
# 5.2.1 Tạo file REST_API.py (Flask REST API Server)
flask_server_code = """import pickle
import os
import numpy as np
import pandas as pd
from flask import Flask, request, jsonify

app = Flask(__name__)

# --- Đường dẫn nạp mô hình đã lưu ---
filename = os.path.join(os.path.dirname(__file__), 'models', 'diabetes.sav')
if not os.path.exists(filename):
    filename = 'diabetes.sav'

with open(filename, 'rb') as f:
    artifact = pickle.load(f)

loaded_pipeline = artifact['pipeline']
feature_columns = artifact['feature_columns']
gender_map = artifact.get('gender_map', {'Female': 0, 'Male': 1, 'Other': 2})
smoking_map = artifact.get('smoking_map', {'No Info': 0, 'never': 1, 'former': 2, 'not current': 3, 'ever': 4, 'current': 5})

@app.route('/diabetes/v1/predict', methods=['POST'])
def predict():
    # --- Nhận dữ liệu JSON từ client ---
    features = request.json
    if not features:
        return jsonify({'error': 'No input data provided'}), 400
    
    # Hỗ trợ cả định dạng cơ bản (BMI, Age, Glucose) lẫn định dạng đầy đủ
    age = float(features.get("Age", features.get("age", 45)))
    bmi = float(features.get("BMI", features.get("bmi", 25.0)))
    glucose = float(features.get("Glucose", features.get("blood_glucose_level", 100)))
    hba1c = float(features.get("HbA1c", features.get("hbA1c_level", 5.5)))
    gender_raw = features.get("Gender", features.get("gender", "Female"))
    gender = gender_map.get(gender_raw, 0) if isinstance(gender_raw, str) else int(gender_raw)
    hypertension = int(features.get("hypertension", 0))
    heart_disease = int(features.get("heart_disease", 0))
    smoking_raw = features.get("smoking_history", "never")
    smoking = smoking_map.get(smoking_raw, 1) if isinstance(smoking_raw, str) else int(smoking_raw)
    race = features.get("race", "Caucasian")
    
    race_cols = {f'race:{r}': 0 for r in ['AfricanAmerican', 'Asian', 'Caucasian', 'Hispanic', 'Other']}
    race_key = f'race:{race}'
    if race_key in race_cols:
        race_cols[race_key] = 1
        
    input_row = {
        'gender': gender,
        'age': age,
        **race_cols,
        'hypertension': hypertension,
        'heart_disease': heart_disease,
        'smoking_history': smoking,
        'bmi': bmi,
        'hbA1c_level': hba1c,
        'blood_glucose_level': glucose
    }
    
    df_input = pd.DataFrame([input_row])[feature_columns]
    
    # --- Thực hiện dự đoán ---
    prediction = loaded_pipeline.predict(df_input)
    confidence = loaded_pipeline.predict_proba(df_input)
    
    # --- Đóng gói phản hồi JSON ---
    response = {}
    response['prediction'] = int(prediction[0])
    response['label'] = "Diabetic" if int(prediction[0]) == 1 else "Not Diabetic"
    response['confidence'] = str(round(np.amax(confidence[0]) * 100, 2))
    return jsonify(response)

if __name__ == '__main__':
    print("Flask API Server is running on port 5000...")
    app.run(host='0.0.0.0', port=5000)
"""

with open('REST_API.py', 'w', encoding='utf-8') as f:
    f.write(flask_server_code)

print("Đã tạo thành công file 'REST_API.py' theo chuẩn giáo trình!")

### 5.3 Kiểm thử REST API bằng lệnh `cURL`

Khi dịch vụ Flask đang chạy (`python REST_API.py`), bạn có thể mở Terminal và kiểm thử bằng lệnh cURL:

**Lệnh cURL trên Linux/macOS:**
```bash
curl -H "Content-type: application/json" -X POST http://127.0.0.1:5000/diabetes/v1/predict -d '{"BMI":30, "Age":29, "Glucose":100}'
```

**Lệnh cURL trên Windows Command Prompt (Lưu ý dấu escape `\"` cho chuỗi JSON):**
```cmd
curl -H "Content-type: application/json" -X POST http://127.0.0.1:5000/diabetes/v1/predict -d "{"BMI":30, "Age":29, "Glucose":100}"
```

**Kết quả phản hồi kỳ vọng:**
```json
{"confidence":"95.42", "label":"Not Diabetic", "prediction":0}
```

### 5.4 Tạo Ứng dụng Client Python (`Predict_Diabetes.py`)

In [ ]:
# 5.4.1 Tạo file Client 'Predict_Diabetes.py'
client_code = """import json
import requests

def predict_diabetes(BMI, Age, Glucose, HbA1c=5.5):
    url = 'http://127.0.0.1:5000/diabetes/v1/predict'
    data = {
        "BMI": float(BMI),
        "Age": int(Age),
        "Glucose": float(Glucose),
        "HbA1c": float(HbA1c)
    }
    data_json = json.dumps(data)
    headers = {'Content-type': 'application/json'}
    try:
        response = requests.post(url, data=data_json, headers=headers)
        result = json.loads(response.text)
        return result
    except Exception as e:
        return {"error": str(e)}

if __name__ == "__main__":
    print("=== CHƯƠNG TRÌNH DỰ ĐOÁN NGUY CƠ TIỂU ĐƯỜNG (CLIENT) ===")
    BMI = input('Nhập chỉ số BMI (ví dụ: 28.5): ') or 28.5
    Age = input('Nhập độ tuổi (ví dụ: 45): ') or 45
    Glucose = input('Nhập chỉ số Glucose mg/dL (ví dụ: 150): ') or 150
    
    predictions = predict_diabetes(BMI, Age, Glucose)
    if "error" in predictions:
        print("Lỗi kết nối:", predictions["error"])
    else:
        status = "Diabetic (Có nguy cơ)" if predictions["prediction"] == 1 else "Not Diabetic (Không có nguy cơ)"
        print(f"\nKết quả chẩn đoán: {status}")
        print(f"Độ tin cậy:        {predictions['confidence']}%")
"""

with open('Predict_Diabetes.py', 'w', encoding='utf-8') as f:
    f.write(client_code)

print("Đã tạo thành công file Client 'Predict_Diabetes.py'!")

In [ ]:
# 5.4.2 Mô phỏng hàm gọi Client trực tiếp trong Notebook (In-Memory Validation)
def local_predict(bmi=28.5, age=45, glucose=150, hba1c=7.0):
    input_data = pd.DataFrame([{
        'gender': 1, 'age': age,
        'race:AfricanAmerican': 0, 'race:Asian': 0, 'race:Caucasian': 1,
        'race:Hispanic': 0, 'race:Other': 0,
        'hypertension': 0, 'heart_disease': 0, 'smoking_history': 1,
        'bmi': bmi, 'hbA1c_level': hba1c, 'blood_glucose_level': glucose
    }])[loaded_artifact['feature_columns']]
    
    pred = loaded_pipe.predict(input_data)[0]
    conf = np.amax(loaded_pipe.predict_proba(input_data)[0]) * 100
    
    print(f"Thông số: BMI={bmi}, Age={age}, Glucose={glucose} mg/dL, HbA1c={hba1c}%")
    print(f"  -> Kết quả: {'CÓ NGUY CƠ TIỂU ĐƯỜNG' if pred == 1 else 'KHÔNG CÓ NGUY CƠ'}")
    print(f"  -> Độ tin cậy: {conf:.2f}%\n")

print("Demo dự đoán bệnh nhân 1 (Chỉ số bình thường):")
local_predict(bmi=22.0, age=30, glucose=95, hba1c=5.2)

print("Demo dự đoán bệnh nhân 2 (Chỉ số cảnh báo cao):")
local_predict(bmi=33.5, age=58, glucose=180, hba1c=7.5)

---
### ✅ TỔNG KẾT VÀ HƯỚNG DẪN VẬN HÀNH TOÀN DIỆN

1. **Khởi chạy Flask REST API (Port 5000)**:
   ```bash
   python REST_API.py
   ```
2. **Khởi chạy Python Client tương tác**:
   ```bash
   python Predict_Diabetes.py
   ```
3. **Khởi chạy FastAPI Server & Web UI (Port 8001)**:
   ```bash
   uvicorn app:app --reload --port 8001
   ```
   Truy cập Web UI tại: `http://localhost:8001`